# 沪深300 PE估值策略 分析笔记本
每月运行一次，查看当前信号和历史表现。

In [1]:
# ── 0. 依赖检查 ───────────────────────────────────────────
import subprocess, sys
pkgs = ['akshare', 'pandas', 'numpy', 'matplotlib']
for p in pkgs:
    try:
        __import__(p)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', p])
print('依赖就绪')

依赖就绪


In [1]:
# ── 1. 拉取最新数据 ────────────────────────────────────────
from fetch_data import fetch_price, fetch_pe, load_merged

fetch_price()
fetch_pe()
df = load_merged()
print(f'数据范围：{df["date"].iloc[0].date()} → {df["date"].iloc[-1].date()}，共 {len(df)} 条')

正在获取沪深300价格数据...
价格数据已保存：c:\Users\Lenovo\Desktop\csi300_strategy\data\csi300_price.csv（共 3931 条）
正在获取沪深300 PE数据...
PE数据已保存：c:\Users\Lenovo\Desktop\csi300_strategy\data\csi300_pe.csv（共 3931 条）
数据范围：2010-01-04 → 2026-03-16，共 3931 条


In [2]:
# ── 2. 当前信号 ────────────────────────────────────────────
from strategy import current_signal

sig = current_signal()
print('\n=== 当前信号 ===')
print(f"日期        : {sig['date']}")
print(f"沪深300收盘 : {sig['close']}")
print(f"PE (TTM)    : {sig['pe']}")
print(f"PE历史百分位: {sig['pe_pct']}%")
print(f"▶ 建议仓位  : {int(sig['position']*100)}%")


=== 当前信号 ===
日期        : 2026-03-16
沪深300收盘 : 4671.56
PE (TTM)    : 13.52
PE历史百分位: 92.6%
▶ 建议仓位  : 0%


In [3]:
# ── 3. 回测 ────────────────────────────────────────────────
from backtest import run_backtest, calc_metrics

bt = run_backtest()
metrics = calc_metrics(bt)

print('\n=== 回测绩效 ===')
for k, v in metrics.items():
    print(f'{k}: {v}')


=== 回测绩效 ===
策略_总收益: 85.2%
策略_年化收益: 4.8%
策略_最大回撤: -24.8%
策略_年化波动: 12.8%
策略_夏普比率: 0.43
策略_卡玛比率: 0.19
基准_总收益: 90.3%
基准_年化收益: 5.0%
基准_最大回撤: -46.7%
基准_年化波动: 21.4%
基准_夏普比率: 0.34
基准_卡玛比率: 0.11
回测起始: 2012-06-26
回测结束: 2026-03-16
回测天数: 3332


In [ ]:
# ── 4. 净值曲线 ────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# 修复1：中文字体
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 修复2：确保reports目录存在，用绝对路径保存
REPORT_DIR = Path(r'C:\Users\Lenovo\Desktop\csi300_strategy\reports')
REPORT_DIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
fig.suptitle('沪深300 PE估值策略', fontsize=14, fontweight='bold')

# 净值曲线
ax1 = axes[0]
ax1.plot(bt['date'], bt['strategy_nav'],  label='策略', color='steelblue', linewidth=1.5)
ax1.plot(bt['date'], bt['benchmark_nav'], label='基准（满仓）', color='gray', linewidth=1, alpha=0.7)
ax1.set_ylabel('净值（元）')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 仓位
ax2 = axes[1]
ax2.fill_between(bt['date'], bt['position'], alpha=0.6, color='steelblue')
ax2.set_ylabel('仓位')
ax2.set_ylim(0, 1.1)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{int(y*100)}%'))
ax2.grid(True, alpha=0.3)

# PE百分位
ax3 = axes[2]
ax3.plot(bt['date'], bt['pe_pct'], color='darkorange', linewidth=1)
ax3.axhline(0.2, color='green',  linestyle='--', alpha=0.5, label='20%')
ax3.axhline(0.8, color='red',    linestyle='--', alpha=0.5, label='80%')
ax3.set_ylabel('PE历史百分位')
ax3.set_ylim(0, 1)
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{int(y*100)}%'))
ax3.legend()
ax3.grid(True, alpha=0.3)

ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(REPORT_DIR / 'strategy_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('图表已保存至 reports/strategy_chart.png')

In [ ]:
# ── 5. 回撤分析 ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))

s_dd = (bt['strategy_nav']  / bt['strategy_nav'].cummax()  - 1) * 100
b_dd = (bt['benchmark_nav'] / bt['benchmark_nav'].cummax() - 1) * 100

ax.fill_between(bt['date'], s_dd, label='策略回撤', color='steelblue', alpha=0.6)
ax.fill_between(bt['date'], b_dd, label='基准回撤', color='gray',      alpha=0.4)
ax.set_ylabel('回撤（%）')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.title('回撤对比')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6. （可选）参数优化 ────────────────────────────────────
# 耗时较长，按需运行
# from optimize import run_optimization
# results = run_optimization()
# results.head(10)

In [ ]:
# ── 7. 保存报告 ────────────────────────────────────────────
from report import save_report
save_report(bt, sig, metrics)
print('完成')